# NVIDIA ModelOpt Quantization Aware Training (QAT) Walkthrough

**Quantization Aware Training (QAT)** is a method that simulates the effects of quantization during neural network post-training to preserve accuracy when deploying models in very-low-precision formats. Unlike post-training quantization, QAT inserts "fake quantization" nodes into the computational graph, mimicking the rounding and clamping operations that occur during actual quantization. This allows the model to adapt its weights and activations to mitigate accuracy loss.

This notebook demonstrates how to apply Quantization Aware Training (QAT) to an LLM, Meta's Llama-3.1-8b in this case with NVIDIA's TensorRT Model Optimizer (ModelOpt) QAT toolkit. We walk through downloading and loading the model, calibrating it using an example dataset, specifically CNN/DailyMail dataset sample, applying NVFP4 quantization, generating outputs, and exporting the quantized model.

## Installing Prerequisites and Dependancies

If you haven't already, install the required dependencies for this notebook. Key dependancies include:

- nvidia-modelopt
- torch
- transformers
- jupyterlab

This repo contains a `examples/llm_qat/notebooks/requirements.txt` file that can be used to install all required dependancies.

In [ ]:
!pip install -r requirements.txt

## Setting HuggingFace Token and Model for Download

Set the HF_TOKEN environment variable making sure to update it to include you token (eg. `%env HF_TOKEN=hf_abdxyz...`)

In [ ]:
%env HF_TOKEN=<YOUR_HUGGINGFACE_TOKEN>

As mentioned above, we will use Meta's **Llamma-3.1-8B-Instruct** in this example

In [1]:
model_name = "meta-llama/Llama-3.1-8B-Instruct"

## Import Required Libraries

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from modelopt.torch.utils.dataset_utils import create_forward_loop, get_dataset_dataloader

In [3]:
import modelopt.torch.opt as mto

# Enable automatic save/load of modelopt state huggingface checkpointing
# modelopt state will be saved automatically to "modelopt_state.pth"
mto.enable_huggingface_checkpointing()

ModelOpt save/restore enabled for `transformers` library.
ModelOpt save/restore enabled for `diffusers` library.
ModelOpt save/restore enabled for `peft` library.


## Model Configuration

Configure the model parameters including the model path, attention implementation, and data type. Set up the model configuration and prepare the model loading arguments.

In [5]:
from transformers import AutoConfig, Mxfp4Config
from trl import ModelConfig

model_args = ModelConfig(
    model_name_or_path="meta-llama/Llama-3.1-8B-Instruct",
    attn_implementation="eager",
    torch_dtype="bfloat16",
)
model_kwargs = {
    "revision": model_args.model_revision,
    "trust_remote_code": model_args.trust_remote_code,
    "attn_implementation": model_args.attn_implementation,
    "torch_dtype": model_args.torch_dtype,
    "use_cache": False,
    "device_map": "auto",
}

# Dequantize if the model is in MXFP4 format (for gpt-oss family of models)
config = AutoConfig.from_pretrained(model_args.model_name_or_path)
if (
    getattr(config, "quantization_config", {})
    and config.quantization_config.get("quant_method", None) == "mxfp4"
):
    model_kwargs["quantization_config"] = Mxfp4Config(dequantize=True)

## Load the Model and Tokenizer

Load the pre-trained model and tokenizer with the specified configuration.

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(model_args.model_name_or_path, **model_kwargs)

tokenizer = AutoTokenizer.from_pretrained(
    model_args.model_name_or_path,
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## Dataset Configuration

Set up the dataset parameters for training and evaluation. This includes specifying the dataset name, train/test splits, and test size ratio.

In [7]:
from trl import ScriptArguments

script_args = ScriptArguments(
    dataset_name="HuggingFaceH4/Multilingual-Thinking",
    dataset_train_split="train",
    dataset_test_split="test",
)
test_size = 0.1

## Load and Prepare Dataset

Load the dataset and split it into training and evaluation sets. The dataset is split with the specified test size ratio and random seed for reproducibility.

In [8]:
from datasets import load_dataset

dataset = load_dataset(script_args.dataset_name)
# split the dataset into train and test
dataset = dataset[script_args.dataset_train_split].train_test_split(test_size=test_size, seed=42)
train_dataset = dataset[script_args.dataset_train_split]
eval_dataset = dataset[script_args.dataset_test_split]

## Training Configuration

Configure the training parameters including epochs, batch sizes, learning rate, gradient accumulation, and evaluation strategy. This sets up the SFT configuration for supervised fine-tuning.

In [9]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="llama3.1-8b-instruct-multilingual-reasoner",
    num_train_epochs=0.1,
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    max_length=4096,
    warmup_ratio=0.03,
    eval_strategy="steps",
    eval_on_start=True,
    logging_steps=10,
    save_steps=50,
    eval_steps=10,
    save_total_limit=2,
)

## Initialize Trainer

Set up the SFT trainer with the model, dataset, and training configuration.

In [10]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset[script_args.dataset_train_split],
    eval_dataset=dataset[script_args.dataset_test_split],
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

## Quantization aware Training

Configure the quantization parameters and prepare the calibration dataset. This step sets up the quantization configuration, creates a calibration subset from the evaluation dataset, and defines a forward loop function for model calibration. The calibration process helps determine optimal quantization scales for the model weights and activations.

In [11]:
import torch

import modelopt.torch.quantization as mtq

# MXFP4_MLP_WEIGHT_ONLY_CFG doesn't need calibration, but other quantization configurations may require it.
quantization_config = mtq.NVFP4_DEFAULT_CFG
calib_size = 128

dataset = torch.utils.data.Subset(
    trainer.eval_dataset, list(range(min(len(trainer.eval_dataset), calib_size)))
)
data_loader = trainer.get_eval_dataloader(dataset)


def forward_loop(model):
    for data in data_loader:
        model(**data)

Apply quantization to the model using the prepared configuration and calibration data.

In [12]:
mtq.quantize(model, quantization_config, forward_loop)

Registered <class 'transformers.models.llama.modeling_llama.LlamaAttention'> to _QuantAttention for KV Cache quantization
Inserted 771 quantizers


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): QuantLlamaAttention(
          (q_proj): QuantLinear(
            in_features=4096, out_features=4096, bias=False
            (input_quantizer): TensorQuantizer((2, 1) bit fake block_sizes={-1: 16, 'type': 'dynamic', 'scale_bits': (4, 3)}, amax=8.3750 calibrator=MaxCalibrator quant)
            (output_quantizer): TensorQuantizer(disabled)
            (weight_quantizer): TensorQuantizer((2, 1) bit fake block_sizes={-1: 16, 'type': 'dynamic', 'scale_bits': (4, 3)}, amax=0.7461 calibrator=MaxCalibrator quant)
          )
          (k_proj): QuantLinear(
            in_features=4096, out_features=1024, bias=False
            (input_quantizer): TensorQuantizer((2, 1) bit fake block_sizes={-1: 16, 'type': 'dynamic', 'scale_bits': (4, 3)}, amax=8.3750 calibrator=MaxCalibrator quant)
            (output_quantizer): TensorQuantizer(di

In [13]:
trainer.train()

Step,Training Loss,Validation Loss
0,No log,2.199608
10,1.432400,1.132420
20,1.278100,1.088682
30,1.019100,1.065277
40,1.108500,1.057330


Saved ModelOpt state to llama3.1-8b-instruct-multilingual-reasoner/checkpoint-45/modelopt_state.pth


TrainOutput(global_step=45, training_loss=1.1997137069702148, metrics={'train_runtime': 141.1602, 'train_samples_per_second': 0.638, 'train_steps_per_second': 0.319, 'total_flos': 2055959104045056.0, 'train_loss': 1.1997137069702148})

## Deploying the QAT Model with TensorRT-LLM

In [59]:
%env TRTLLM_SERVE_CMD=trtllm-serve /app/tensorrt_llm/qat/checkpoint-45 --max_batch_size 1 --max_num_tokens 512 --max_seq_len 4096 --tp_size 8 --pp_size 1 --host 0.0.0.0 --port 8000 --kv_cache_free_gpu_memory_fraction 0.95

env: TRTLLM_SERVE_CMD=trtllm-serve /app/tensorrt_llm/qat/checkpoint-45 --max_batch_size 1 --max_num_tokens 512 --max_seq_len 4096 --tp_size 8 --pp_size 1 --host 0.0.0.0 --port 8000 --kv_cache_free_gpu_memory_fraction 0.95


In [60]:
%%sh
docker run --rm --ipc=host -d \
  --ulimit stack=67108864   --ulimit memlock=-1 \
  --gpus all   -p 8000:8000   -e TRTLLM_ENABLE_PDL=1 \
  -v ~/.cache:/root/.cache:rw --name tensorrt_llm \
  -v /home/$USER/qat/llama3.1-8b-instruct-multilingual-reasoner/:/app/tensorrt_llm/qat \
  nvcr.io/nvidia/tensorrt-llm/release:1.1.0rc1  $TRTLLM_SERVE_CMD

9b8741a1681ab6e054f533ea40cd65b51967783b41611fdb722df188ec77d31c


In [61]:
!docker logs tensorrt_llm


== PyTorch ==

NVIDIA Release 25.06 (build 177567386)
PyTorch Version 2.8.0a0+5228986
Container image Copyright (c) 2025, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
Copyright (c) 2014-2024 Facebook Inc.
Copyright (c) 2011-2014 Idiap Research Institute (Ronan Collobert)
Copyright (c) 2012-2014 Deepmind Technologies    (Koray Kavukcuoglu)
Copyright (c) 2011-2012 NEC Laboratories America (Koray Kavukcuoglu)
Copyright (c) 2011-2013 NYU                      (Clement Farabet)
Copyright (c) 2006-2010 NEC Laboratories America (Ronan Collobert, Leon Bottou, Iain Melvin, Jason Weston)
Copyright (c) 2006      Idiap Research Institute (Samy Bengio)
Copyright (c) 2001-2004 Idiap Research Institute (Ronan Collobert, Samy Bengio, Johnny Mariethoz)
Copyright (c) 2015      Google Inc.
Copyright (c) 2015      Yangqing Jia
Copyright (c) 2013-2016 The Caffe contributors
All rights reserved.

Various files include modifications (c) NVIDIA CORPORATION & AFFILIATES.  All rights reserved.

GOVERNI

In [63]:
!curl -s -o /dev/null -w "Status: %{http_code}\n" "http://localhost:8000/health"

Status: 200


In [62]:
%%sh
curl localhost:8000/v1/chat/completions -H "Content-Type: application/json" -d '{
    "model": "meta-llama/Llama-3.1-8B-Instruct-QAT",
    "messages": [
        {
            "role": "user",
            "content": "What is NVIDIAs advantage for inference?"
        }
    ],
    "max_tokens": 1024,
    "top_p": 0.9
}' -w "\n"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3521  100  3285  100   236   3206    230  0:00:01  0:00:01 --:--:--  3438


{"id":"chatcmpl-69bb9c6c65da4b309e61e52d695228f7","object":"chat.completion","created":1756434401,"model":"meta-llama/Llama-3.1-8B-Instruct-QAT","choices":[{"index":0,"message":{"role":"assistant","content":"NVIDIA has several advantages for inference, which is the process of using a trained model to make predictions on new, unseen data. Here are some key advantages:\n\n1. **Tensor Cores**: NVIDIA's Tensor Cores are specialized hardware units designed specifically for matrix multiplications, which are a key component of deep learning models. They provide a significant boost in performance and efficiency compared to traditional CPU or GPU cores.\n\n2. **Mixed-Precision Training**: NVIDIA supports mixed-precision training, which allows models to be trained using lower-precision data types (e.g., FP16) while maintaining high accuracy. This reduces memory usage and speeds up training. For inference, models can be optimized to use even lower precision (e.g., INT8) without significant loss i

In [34]:
%env TRTLLM_SERVE_CMD=trtllm-serve meta-llama/Llama-3.1-8B-Instruct --max_batch_size 1 --max_num_tokens 512 --max_seq_len 4096 --tp_size 8 --pp_size 1 --host 0.0.0.0 --port 8000 --kv_cache_free_gpu_memory_fraction 0.95


env: TRTLLM_CMD=trtllm-serve meta-llama/Llama-3.1-8B-Instruct --max_batch_size 1 --max_num_tokens 512 --max_seq_len 4096 --tp_size 8 --pp_size 1 --host 0.0.0.0 --port 8000 --kv_cache_free_gpu_memory_fraction 0.95


## Deploying Llama3.1-8b-Instruct (original) Model

In [35]:
%%sh
docker run --rm --ipc=host -d \
  --ulimit stack=67108864   --ulimit memlock=-1 \
  --gpus all   -p 8000:8000   -e TRTLLM_ENABLE_PDL=1 \
  -v ~/.cache:/root/.cache:rw --name tensorrt_llm\
  -e HF_TOKEN=$HF_TOKEN
  nvcr.io/nvidia/tensorrt-llm/release:1.1.0rc1  $TRTLLM_SERVE_CMD

fccf586d42986fc7a81b001036686157b98eb2bb93d9b6e014d8eb47fa255591


In [37]:
!docker logs tensorrt_llm

In [31]:
%%sh
curl localhost:8000/v1/chat/completions -H "Content-Type: application/json" -d '{
    "model": "meta-llama/Llama-3.1-8B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": "What is NVIDIAs advantage for inference?"
        }
    ],
    "max_tokens": 1024,
    "top_p": 0.9
}' -w "\n"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3114  100  2882  100   232   3115    250 --:--:-- --:--:-- --:--:--  3366


{"id":"chatcmpl-f7431cdd6afe4a95acd74b9d197583f3","object":"chat.completion","created":1756431947,"model":"meta-llama/Llama-3.1-8B-Instruct","choices":[{"index":0,"message":{"role":"assistant","content":"NVIDIA has several advantages for inference:\n\n1. **Tensor Cores**: NVIDIA's Tensor Cores are a key component of their GPUs, designed specifically for deep learning inference. They provide a significant boost in performance and power efficiency compared to traditional CUDA cores. Tensor Cores can perform matrix multiplications, which are a fundamental operation in deep learning, at a much higher rate than traditional cores.\n\n2. **Mixed-Precision Training and Inference**: NVIDIA's mixed-precision training and inference capabilities allow models to be trained and run at lower precision (e.g., 16-bit or 8-bit) while maintaining high accuracy. This reduces memory bandwidth requirements and increases performance.\n\n3. **NVIDIA Deep Learning Accelerator (NVDLA)**: NVDLA is a hardware acc

## Stop the TensorRT-LLM Docker contrainer

In [64]:
!docker stop tensorrt_llm

tensorrt_llm
